# Your first virtual photonics table

**Lesson:** `photonics.polarization.01`, revision 1 · development teaching material · GPL-3.0-only.

**Outcomes:** predict and measure Malus’ law; distinguish a wave plate from a polarizer;
understand Gaussian diffraction; steer a beam using an ideal phase SLM; save reproducible results.

**Prerequisites:** complex numbers, vectors, sine/cosine, basic Python. A coherent monochromatic
beam is a classical field, not a single-photon or entanglement simulation.

This notebook runs locally without credentials or a GPU. Start from the locked PhotonicsLab
notebook environment. The arrays are small (at most 256 × 256 × 2 complex128 samples here);
execution should take seconds, with no hardware calls. Numeric assertions are the assessment
checks for this development revision; instructor/science review remains necessary before publication.


## Predict before running

All lengths are in metres, angles in radians, power in watts. The complex envelope uses
$\mathrm{Re}[E\exp(ikz-i\omega t)]$ and components $(E_x,E_y)$.

An analyser at angle $\alpha$ projects onto $u=(\cos\alpha,\sin\alpha)$:
$P_\alpha=uu^T$. A retarder at angle $\theta$ acts as
$J=uu^T+e^{i\delta}vv^T$, where $v=(-\sin\theta,\cos\theta)$.

For a horizontal input and a half-wave plate ($\delta=\pi$), the output is linear
at $2\theta$. A horizontal analyser therefore transmits
$T=\cos^2(2\theta)$. Predict the readings at 0°, 22.5° and 45° before running.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from photonicslab import (
    Grid, GaussianBeam, OpticalBench, HalfWavePlate, QuarterWavePlate,
    LinearPolarizer, ThinLens, FreeSpace, PhaseSLM,
)

grid = Grid(nx=128, ny=96, width=4e-3, height=3e-3)
laser = GaussianBeam(wavelength=632.8e-9, waist=0.35e-3, power=1e-3)
beam = laser.emit(grid)
bench = OpticalBench([HalfWavePlate(angle=np.pi/8), LinearPolarizer(angle=0)])
result = bench.run(beam)
transmission = result.output.power / beam.power
assert abs(transmission - 0.5) < 1e-12
print(f"Input: {beam.power*1e3:.3f} mW; output: {result.output.power*1e3:.3f} mW")


In [ ]:
angles = np.linspace(0, np.pi/2, 37)
measured = np.array([
    OpticalBench([HalfWavePlate(a), LinearPolarizer()]).run(beam).output.power / beam.power
    for a in angles
])
predicted = np.cos(2*angles)**2
assert np.max(np.abs(measured-predicted)) < 1e-12
fig, ax = plt.subplots()
ax.plot(np.rad2deg(angles), predicted, label="Theory: cos²(2θ)")
ax.scatter(np.rad2deg(angles), measured, s=12, label="Virtual power meter")
ax.set(xlabel="Half-wave plate angle θ (degrees)", ylabel="Transmitted power / incident power")
ax.legend(); plt.show()


## A quarter-wave plate changes relative phase

Integrated Stokes parameters have units of watts. Here
$S_3=2\operatorname{Im}\int E_x^*E_y\,dA$; we specify this formula rather than
relying on conflicting right/left circular naming conventions. A QWP at 45° gives
$(S_0,S_1,S_2,S_3)/P=(1,0,0,-1)$ with this retarder and time convention.


In [ ]:
circular = QuarterWavePlate(np.pi/4).apply(beam)
np.testing.assert_allclose(circular.stokes / beam.power, [1, 0, 0, -1], atol=1e-12)
print("Normalized Stokes:", circular.stokes / beam.power)


## Diffraction and a focusing lens

For a Gaussian waist $w_0$, $z_R=\pi n w_0^2/\lambda_0$. Free propagation obeys
$w(z)=w_0\sqrt{1+(z/z_R)^2}$. A lens followed by its focal distance $f$ gives
$w(f)=\lambda_0 f/(\pi n w_0)$ for an incident field at its waist.

`FreeSpace` uses an FFT window with periodic boundaries. Enlarge the window and refine the
sampling to check convergence. Conserved norm alone does not prove an accurate optical simulation.
The angular-spectrum option is scalar propagation of each Jones component, not a high-NA vector solver.


In [ ]:
fine_grid = Grid(256, 256)
fine_beam = laser.emit(fine_grid)
focal_length = 0.2
focused = OpticalBench([ThinLens(focal_length), FreeSpace(focal_length)]).run(fine_beam).output
weights = focused.intensity.sum(axis=0)
centre = np.sum(fine_grid.x*weights)/weights.sum()
measured_radius = 2*np.sqrt(np.sum((fine_grid.x-centre)**2*weights)/weights.sum())
predicted_radius = laser.wavelength*focal_length/(np.pi*laser.waist)
assert abs(measured_radius/predicted_radius-1) < 1e-5
assert abs(focused.power/fine_beam.power-1) < 1e-12
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, field, title in zip(axes, [fine_beam, focused], ["Before lens", "At focal plane"]):
    image = ax.imshow(field.intensity, extent=[*fine_grid.x[[0,-1]]*1e3, *fine_grid.y[[0,-1]]*1e3], origin="lower")
    ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)")
    fig.colorbar(image, ax=ax, label="Irradiance (W/m²)")
plt.tight_layout(); plt.show()
print(f"Predicted/measured radius: {predicted_radius*1e6:.3f}/{measured_radius*1e6:.3f} µm")


## An SLM is a programmable phase element

An ideal horizontal-axis phase SLM multiplies $E_x$ by $e^{i\phi(x,y)}$ and leaves
$E_y$ unchanged. A phase ramp $\phi=2\pi f_x x$ steers the beam: in the paraxial
limit its centroid shifts by $\Delta x=z\lambda_0 f_x/n$.
A real device also needs wavelength-specific calibration, a pixel pitch/fill factor,
phase quantisation and polarization alignment; those effects are not modelled here.


In [ ]:
small_beam = GaussianBeam(waist=0.12e-3).emit(fine_grid)
fx, distance = 10/fine_grid.width, 0.2
slm = PhaseSLM(2*np.pi*fx*fine_grid.mesh[0])
steered = OpticalBench([slm, FreeSpace(distance)]).run(small_beam).output
weights = steered.intensity.sum(axis=0)
centroid = np.sum(fine_grid.x*weights)/weights.sum()
predicted_centroid = distance*small_beam.wavelength*fx
assert abs(centroid-predicted_centroid) < 1e-9
print(f"SLM shift: {centroid*1e3:.3f} mm")


## Translate the model to a real teaching bench

The corresponding layout is a teaching laser → wave plate on a rotation mount → analyser → power meter.
Use the measured incident power as the reference. Sweep the mount angle, subtract the detector dark
reading, and compare the normalized measurements to cos²(2θ). Fit a zero-angle offset before concluding
that the physics is wrong. Finite extinction, imperfect retardance, detector uncertainty and alignment
explain deviations that this ideal model does not include.

For the SLM lesson, compare the measured diffraction angle to the phase-ramp prediction, using the
actual pixel pitch and a calibrated phase response. Hardware access is a separate, instructor-controlled
adapter, never an automatic consequence of executing this notebook. Follow the institution’s supervised
laser procedure; the virtual power setting is not a physical laser safety assessment.

**Exercises:**

1. Predict, then verify, a 45° analyser between a horizontal beam and a vertical analyser (T = 1/4).
2. Repeat the lens calculation on a larger window and a finer grid. Which change resolves clipping,
   and which resolves undersampling?
3. Change the SLM-active axis to vertical. Explain the outcome for the horizontal input.
4. Explain why these Jones calculations cannot demonstrate antibunching or SPDC photon-pair statistics.


In [ ]:
crossed_with_middle = OpticalBench([
    LinearPolarizer(np.pi/4), LinearPolarizer(np.pi/2),
]).run(beam).output
assert abs(crossed_with_middle.power/beam.power - 0.25) < 1e-12

from photonicslab.artifacts import run_polarization_lab
summary, run_directory = run_polarization_lab(Path("results"), angle=np.pi/8, samples=64)
assert summary["passed"]
print("Local lesson artifacts:", run_directory)


## References and next steps

- J. W. Goodman, *Introduction to Fourier Optics*, 4th edition, W. H. Freeman, 2017.
- B. E. A. Saleh and M. C. Teich, *Fundamentals of Photonics*, 3rd edition, Wiley, 2019.
- Primary software comparisons and links: [ecosystem review](../../docs/ECOSYSTEM.md).
- Solver assumptions and staged research plan: [architecture](../../docs/ARCHITECTURE.md).

The next lessons should add interferometry and calibrated nonideal components before introducing
cavity/gain solvers or quantum optical states. This notebook is original project material; it does not
embed textbook figures or redistribute third-party course notebooks.
